# Generate Video Audio (Local — Bark)

Generates spoken audio from text you provide and adds it to any video, in any supported language.
Any existing audio in the video is **stripped and replaced**.
Uses Suno's [Bark](https://github.com/suno-ai/bark) model via HuggingFace — runs fully on GPU, **no API key needed**.

**Setup:** Mount Google Drive to cache model weights (~5 GB) and store videos.

**Usage:** Set `VIDEO_PATH`, `TEXT`, and `VOICE_PRESET`, then run all cells.

In [ ]:
# Mount Google Drive and install dependencies
from google.colab import drive
drive.mount("/content/drive")

!pip install -q transformers torch torchaudio ffmpeg-python scipy optimum accelerate

In [ ]:
# ── Config ──────────────────────────────────────────────────────────
VIDEO_PATH = "/content/drive/MyDrive/videos/input.mp4"  # input video (original audio will be replaced)
MODEL_CACHE = "/content/drive/MyDrive/models/bark"  # weights cache

# The text to speak over the video (in the target language)
TEXT = """
Your narration text goes here. Write it in the language you want spoken.
""".strip()

# Voice preset: "v2/{lang}_speaker_{id}" where id is 0–9
# Bark auto-detects language from text, but the preset controls the speaker's accent/voice.
VOICE_PRESET = "v2/en_speaker_6"

# Available language prefixes for voice presets:
# fmt: off
VOICE_LANGUAGES = {
    "en": "English",    "zh": "Chinese",    "fr": "French",
    "de": "German",     "hi": "Hindi",      "it": "Italian",
    "ja": "Japanese",   "ko": "Korean",     "pl": "Polish",
    "pt": "Portuguese", "ru": "Russian",    "es": "Spanish",
    "tr": "Turkish",
}
# fmt: on
# Example: for Hindi speaker 3 → "v2/hi_speaker_3"
#          for Japanese speaker 0 → "v2/ja_speaker_0"

assert TEXT, "Provide the text to narrate"
print(f"Voice preset: {VOICE_PRESET}")
print(f"Text length: {len(TEXT)} chars")

In [ ]:
import os, re, torch, torchaudio, tempfile, numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoProcessor, BarkModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load model (cached on Drive so subsequent runs are instant)
print("Loading Bark model (first run downloads ~5 GB to Drive)...")
os.makedirs(MODEL_CACHE, exist_ok=True)

processor = AutoProcessor.from_pretrained("suno/bark", cache_dir=MODEL_CACHE)
model = BarkModel.from_pretrained(
    "suno/bark", cache_dir=MODEL_CACHE, torch_dtype=torch.float16,
).to(device)

SAMPLE_RATE = model.generation_config.sample_rate  # 24000 Hz
print(f"Model loaded. Sample rate: {SAMPLE_RATE} Hz")

In [ ]:
import ffmpeg


def split_into_sentences(text):
    """Split text into sentences for chunk-by-chunk generation.
    Bark handles ~13s of audio per generation, roughly 1-2 sentences."""
    # Split on sentence-ending punctuation followed by whitespace
    parts = re.split(r'(?<=[.!?;\u3002\uff01\uff1f])\s+', text.strip())
    # Merge very short fragments with the previous sentence
    sentences = []
    for p in parts:
        p = p.strip()
        if not p:
            continue
        if sentences and len(sentences[-1]) < 20:
            sentences[-1] += " " + p
        else:
            sentences.append(p)
    return sentences


def synthesize(text, voice_preset):
    """Generate speech from text using Bark, sentence by sentence."""
    sentences = split_into_sentences(text)
    print(f"  Split into {len(sentences)} sentence(s)")

    audio_parts = []
    # Small silence gap between sentences (0.15s)
    silence = np.zeros(int(SAMPLE_RATE * 0.15), dtype=np.float32)

    for sent in tqdm(sentences, desc="Synthesizing", unit="sentence"):
        inputs = processor(sent, voice_preset=voice_preset, return_tensors="pt").to(device)
        with torch.no_grad():
            output = model.generate(**inputs)
        audio = output.cpu().numpy().squeeze()
        audio_parts.append(audio)
        audio_parts.append(silence)

    # Remove trailing silence
    if audio_parts and len(audio_parts) > 1:
        audio_parts = audio_parts[:-1]

    return np.concatenate(audio_parts)


def add_audio_to_video(video_path, audio_np, output_path, sr):
    """Strip original audio, mux video with generated audio, adjusting tempo to match video duration."""
    tmp_wav = tempfile.mktemp(suffix=".wav")
    audio_tensor = torch.from_numpy(audio_np).unsqueeze(0).float()
    torchaudio.save(tmp_wav, audio_tensor, sr)

    video_dur = float(ffmpeg.probe(video_path)["format"]["duration"])
    audio_dur = len(audio_np) / sr

    print(f"  Video duration: {video_dur:.1f}s")
    print(f"  Audio duration: {audio_dur:.1f}s")

    # Select only the video stream — any existing audio is discarded
    vid = ffmpeg.input(video_path).video
    aud = ffmpeg.input(tmp_wav)

    # Adjust audio speed to match video length if within a reasonable range
    tempo = audio_dur / video_dur
    if 0.5 <= tempo <= 2.0 and abs(tempo - 1.0) > 0.05:
        print(f"  Adjusting audio tempo by {tempo:.2f}x to match video length")
        aud_stream = aud.audio.filter("atempo", tempo)
    else:
        if tempo > 2.0 or tempo < 0.5:
            print(f"  Warning: audio/video duration ratio ({tempo:.2f}x) is too extreme to adjust. Audio may not match video length.")
        aud_stream = aud.audio

    ffmpeg.output(
        vid, aud_stream, output_path,
        vcodec="copy", acodec="aac", strict="experimental",
    ).overwrite_output().run(quiet=True)
    os.remove(tmp_wav)


print("Functions ready.")

In [ ]:
# ── Generate ────────────────────────────────────────────────────────
video = Path(VIDEO_PATH)
assert video.exists(), f"Video not found: {VIDEO_PATH}"

lang = VOICE_PRESET.split("/")[-1].split("_")[0] if "/" in VOICE_PRESET else "narr"
output_path = str(video.with_stem(f"{video.stem}_{lang}_narrated"))

print(f"Input:  {VIDEO_PATH}")
print(f"Output: {output_path}\n")

progress = tqdm(total=2, desc="Progress", unit="step")

# Step 1: Synthesize speech from text
progress.set_postfix_str("synthesizing speech")
audio = synthesize(TEXT, VOICE_PRESET)
print(f"Synthesized audio: {len(audio) / SAMPLE_RATE:.1f}s")
progress.update(1)

# Step 2: Add audio to video
progress.set_postfix_str("muxing audio with video")
add_audio_to_video(VIDEO_PATH, audio, output_path, SAMPLE_RATE)
progress.update(1)

progress.set_postfix_str("done")
progress.close()
print(f"\nVideo with narration saved to:\n{output_path}")